<a href="https://colab.research.google.com/github/entropy-om/entheai/blob/rahul-phi-work/Rahul_rangarao_phi_fixed.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<a href="https://colab.research.google.com/github/entropy-om/entheai/blob/rahul-phi-work/Rahul_rangarao_phi.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# rahul-haiku: QLoRA fine-tune of Phi-4-mini-instruct

This notebook was cleaned up to remove a recurring bug: several earlier cells replaced
`model.lm_head` with a randomly-initialized `CustomLMHead` (for a one-off token-sampling
demo) and that replacement kept getting re-attached right before training. Since the custom
head was never trained and wasn't included in `modules_to_save`, LoRA training could never
fix the output projection — that's what caused the near-random loss (~11.7, ≈ln(vocab)) and
the recurring `RuntimeError: expected scalar type BFloat16 but found Float` (its `LayerNorm`
clashes with bf16 autocast).

**Rule for this notebook: never reassign `model.lm_head`.** Everything below uses the
pretrained head as-is.

In [12]:
!pip install -q torch transformers datasets peft bitsandbytes trl accelerate


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 885.0/885.0 kB 54.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 555.1/555.1 kB 48.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 48.2 MB/s eta 0:00:00


In [13]:
# Optional: font for wide-unicode plotting, harmless if skipped
!sudo apt-get install -y fonts-wqy-zenhei -qq > /dev/null
import matplotlib.font_manager as fm
fm._load_fontmanager(try_read_cache=False)


debconf: unable to initialize frontend: Dialog
debconf: (No usable dialog-like program is installed, so the dialog based frontend cannot be used. at /usr/share/perl5/Debconf/FrontEnd/Dialog.pm line 78, <> line 1.)
debconf: falling back to frontend: Readline
debconf: unable to initialize frontend: Readline
debconf: (This frontend requires a controlling tty.)
debconf: falling back to frontend: Teletype
dpkg-preconfigure: unable to re-open stdin: 


In [1]:
!pip install -U bitsandbytes==0.49.2

## Setup

In [2]:
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    EarlyStoppingCallback,
)
import torch
import matplotlib.pyplot as plt
import pandas as pd


In [3]:
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


Set this to wherever you uploaded `train.jsonl` / `valid.jsonl` / `test.jsonl` on Drive.

In [6]:
DATA_DIR = "/content/drive/MyDrive/Colab Notebooks/rahul-ai"
ADAPTER_DIR = "/content/adapters"
DRIVE_ADAPTER_DIR = f"{DATA_DIR}/adapters"


## 1. Load the base model (4-bit, bf16 — single explicit dtype, no `"auto"`)

In [5]:
model_id = "microsoft/phi-4-reasoning"

bnb_config = BitsAndBytesConfig(
    load_in_8bit=True,
)

tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto",
)

model.safetensors.index.json:   0%|          | 0.00/20.4k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 6 files:   0%|          | 0/6 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/243 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/222 [00:00<?, ?B/s]

## 2. Baseline check — confirm the untrained model doesn't know "Rahul"

This is just a sanity check, not part of the training path.

In [7]:
messages = [{"role": "user", "content": "Who are you? Who am I?"}]

inputs = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt=True,
    tokenize=True,
    return_dict=True,
    return_tensors="pt",
).to(model.device)

outputs = model.generate(**inputs, max_new_tokens=40)
print(tokenizer.decode(outputs[0][inputs['input_ids'].shape[-1]:], skip_special_tokens=True))


/usr/local/lib/python3.12/dist-packages/bitsandbytes/autograd/_functions.py:123: UserWarning: MatMul8bitLt: inputs will be cast from torch.bfloat16 to float16 during quantization
  warnings.warn(f"MatMul8bitLt: inputs will be cast from {A.dtype} to float16 during quantization")


<think>User question: "Who are you? Who am I?" We have instructions that I am "Phi, a language model developed by Microsoft, trained to provide accurate, secure, and user-aligned responses


Expected: gibberish / generic assistant text. That's the pre-fine-tune baseline.

## 3. LoRA config

In [8]:
from peft import LoraConfig, prepare_model_for_kbit_training

model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=32,
    lora_alpha=64,
    lora_dropout=0.05,
    target_modules=["qkv_proj", "o_proj", "gate_up_proj", "down_proj"],
    task_type="CAUSAL_LM",
)


## 4. Load data

In [10]:
from datasets import load_dataset

data = load_dataset("json", data_files={
    "train": f"{DATA_DIR}/train.jsonl",
    "validation": f"{DATA_DIR}/valid.jsonl",
    "test": f"{DATA_DIR}/test.jsonl",   # keep this private, only touch it once at the very end
})


## 5. Train

Hyperparameters tuned down from earlier runs — `1e-3` overfit past ~step 75 on this dataset
size (train loss kept dropping, validation loss and token accuracy started getting worse).
`load_best_model_at_end` + early stopping mean this cell is safe to just let run — it'll stop
and roll back to the best checkpoint on its own instead of running to the full epoch count.

In [14]:
from trl import SFTTrainer, SFTConfig

sft_config = SFTConfig(
    output_dir=ADAPTER_DIR,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    num_train_epochs=10,
    learning_rate=1e-4,
    eval_strategy="steps",
    eval_steps=25,
    save_strategy="steps",
    save_steps=25,
    logging_steps=10,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    bf16=True,
    packing=False,
)


In [ ]:
trainer = SFTTrainer(
    model=model,
    args=sft_config,
    train_dataset=data["train"],
    eval_dataset=data["validation"],   # test stays untouched until final eval below
    peft_config=lora_config,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)
trainer.train()


Tokenizing train dataset:   0%|          | 0/118 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/118 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/118 [00:00<?, ? examples/s]

Dropping fully masked examples from train dataset:   0%|          | 0/118 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/14 [00:00<?, ? examples/s]

Building labels for eval dataset:   0%|          | 0/14 [00:00<?, ? examples/s]

Truncating eval dataset:   0%|          | 0/14 [00:00<?, ? examples/s]

Dropping fully masked examples from eval dataset:   0%|          | 0/14 [00:00<?, ? examples/s]

Step,Training Loss,Validation Loss,Entropy,Num Tokens,Mean Token Accuracy
25,1.643560,1.660474,1.608357,131829.000000,0.608923
50,1.420967,1.573630,1.487363,257575.000000,0.621903
75,1.262938,1.556030,1.394329,387760.000000,0.624544
100,1.163704,1.569834,1.300567,519179.000000,0.627395
125,1.113222,1.591243,1.251509,648047.000000,0.622734


TrainOutput(global_step=125, training_loss=1.3726893806457519, metrics={'train_runtime': 329.9128, 'train_samples_per_second': 3.577, 'train_steps_per_second': 0.455, 'total_flos': 1.6929093918916608e+16, 'train_loss': 1.3726893806457519, 'epoch': 8.338983050847457})

## 6. Check generation after fine-tuning

In [ ]:
messages = [{"role": "user", "content": "Who are you? Who am I?"}]

inputs = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt=True,
    tokenize=True,
    return_dict=True,
    return_tensors="pt",
).to(model.device)

outputs = model.generate(**inputs, max_new_tokens=40)
print(tokenizer.decode(outputs[0][inputs['input_ids'].shape[-1]:], skip_special_tokens=True))


/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


I am Phi, an AI developed by Microsoft. I am here to help you with a wide range of questions, answer your queries, and assist you with information, explanations, and guidance on many topics


## 7. Final private eval on the held-out test split (do this once, at the end)

In [ ]:
test_metrics = trainer.evaluate(eval_dataset=data["test"])
print(test_metrics)


Tokenizing eval dataset:   0%|          | 0/12 [00:00<?, ? examples/s]

Building labels for eval dataset:   0%|          | 0/12 [00:00<?, ? examples/s]

Truncating eval dataset:   0%|          | 0/12 [00:00<?, ? examples/s]

Dropping fully masked examples from eval dataset:   0%|          | 0/12 [00:00<?, ? examples/s]

Training Loss,Validation Loss,Step,Entropy,Num Tokens,Mean Token Accuracy
1.113222,1.453529,125,1.268677,648047.000000,0.646046


{'eval_loss': 1.4535293579101562, 'eval_entropy': 1.2686767578125, 'eval_num_tokens': 648047.0, 'eval_mean_token_accuracy': 0.6460462808609009}


## 8. Save the adapter to Drive

`/content` disappears when the Colab runtime disconnects — save straight to Drive.

In [ ]:
trainer.save_model(DRIVE_ADAPTER_DIR)
tokenizer.save_pretrained(DRIVE_ADAPTER_DIR)
print(f'Saved to {DRIVE_ADAPTER_DIR}')


Saved to /content/drive/MyDrive/Colab Notebooks/rahul-ai/adapters


## 9. (Optional) Continue training with new hyperparameters

Use this pattern — a **fresh** `SFTTrainer` on the same in-memory `model` — when you want to
change learning rate, optimizer, or epoch count. This intentionally does **not** use
`resume_from_checkpoint`, so a new optimizer/scheduler is built from the new `sft_config`
and your new learning rate actually takes effect (resuming from a checkpoint restores the
old scheduler state, which silently overrides a new LR).

If instead you want a true resume (same hyperparameters, just more steps, preserving
optimizer momentum), call `trainer.train(resume_from_checkpoint="<path-to-checkpoint-N>")`
on the existing `trainer` instead of building a new one.

In [ ]:
sft_config_2 = SFTConfig(
    output_dir=ADAPTER_DIR,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    num_train_epochs=10,
    learning_rate=5e-5,
    optim="paged_adamw_8bit",
    eval_strategy="steps",
    eval_steps=25,
    save_strategy="steps",
    save_steps=25,
    logging_steps=10,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    bf16=True,
    packing=False,
)

trainer = SFTTrainer(
    model=model,              # already has the LoRA weights trained above
    args=sft_config_2,
    train_dataset=data["train"],
    eval_dataset=data["validation"],
    peft_config=lora_config,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)
trainer.train()


/usr/local/lib/python3.12/dist-packages/peft/mapping_func.py:72: UserWarning: You are trying to modify a model with PEFT for a second time. If you want to reload the model with a different config, make sure to call `.unload()` before.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/peft/tuners/tuners_utils.py:302: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(


Step,Training Loss,Validation Loss,Entropy,Num Tokens,Mean Token Accuracy
25,1.762904,1.792244,1.750896,131829.000000,0.589183
50,1.572482,1.666781,1.615962,257575.000000,0.604780
75,1.433817,1.620285,1.578754,387760.000000,0.614001
100,1.392418,1.602013,1.542806,519179.000000,0.614859
125,1.383561,1.595195,1.521103,648047.000000,0.616262
150,1.369814,1.595526,1.518885,775520.000000,0.616439


TrainOutput(global_step=150, training_loss=1.5199794324239095, metrics={'train_runtime': 399.6387, 'train_samples_per_second': 2.953, 'train_steps_per_second': 0.375, 'total_flos': 2.027344028484403e+16, 'train_loss': 1.5199794324239095, 'epoch': 10.0})

In [ ]:
messages = [{"role": "user", "content": "Who are you? Who am I?"}]

inputs = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt=True,
    tokenize=True,
    return_dict=True,
    return_tensors="pt",
).to(model.device)

outputs = model.generate(**inputs, max_new_tokens=40)
print(tokenizer.decode(outputs[0][inputs['input_ids'].shape[-1]:], skip_special_tokens=True))


/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


I am Phi, an AI developed by Microsoft. I am here to help you with a wide range of questions, answer queries, and assist you in finding information. What can I help you with today
